# KCA crossover: P4Transformer seed 2 (full scale)

> **English copy of the executed notebook.** Text, comments and printed messages were translated from Chinese;
> the code is unchanged and the outputs are the ones recorded in the actual run (Oct 22, 2026, Colab).

**What it does**: retrains the two binary task models of the P4Transformer crossover (motion task, shape task) with random seed 2, then explains 40 test clips.
The configuration is identical to Section 6 of `KCA_fullscale_colab.ipynb` (official width, 2048 points per frame); only the seed differs. The motion-task model of seed 1 stayed at chance level (50%), so this run was added.

**How to use**: Runtime → Change runtime type → A100 (L4 also works) → **Runtime → Run all**. The first cell asks for Drive access. About 15 minutes in total.

**The previous results are not overwritten**: it only reads the upload bundle in `MyDrive/kca_full/upload/`; every output goes to the new directory `MyDrive/kca_full/seed2/`. After a disconnect, "Run all" again; finished parts are skipped automatically.

In [1]:
# ===== Configuration =====
import os, sys, glob, json, subprocess, time

DRIVE_ROOT = "/content/drive/MyDrive/kca_full"
UPLOAD = f"{DRIVE_ROOT}/upload"        # read-only: the upload bundle from 10/21 (the crossover code has not changed since)
OUT_ROOT = f"{DRIVE_ROOT}/seed2"       # every output of this notebook (a new directory)
SEED = 2
POINTS = 2048

In [2]:
# Mount Drive; check the GPU and the upload bundle
from google.colab import drive
drive.mount("/content/drive")
import torch
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → choose A100 or L4"
p = torch.cuda.get_device_properties(0)
print(f"GPU {p.name} {p.total_memory / 1e9:.0f} GB | torch {torch.__version__} | CUDA {torch.version.cuda}")
for f in ("kca_bundle.tar.gz", "msr_npz.tar.gz"):
    assert os.path.exists(f"{UPLOAD}/{f}"), f"{UPLOAD}/{f} not found (Colab must be signed in to the Google account that holds kca_full/)"
print("upload bundle ✓")

Mounted at /content/drive
GPU NVIDIA A100-SXM4-40GB 42 GB | torch 2.11.0+cu130 | CUDA 13.0
upload bundle ✓


In [3]:
# Unpack code and data to the local disk, define run()
W = "/content/kca_seed2"
CODE, EXT, DATA = f"{W}/code", f"{W}/ext", f"{W}/data/msr"
LOGS = f"{OUT_ROOT}/logs"
os.makedirs(LOGS, exist_ok=True)
if not os.path.exists(f"{CODE}/src/kca"):
    os.makedirs(W, exist_ok=True)
    subprocess.run(["tar", "-xzf", f"{UPLOAD}/kca_bundle.tar.gz", "-C", W], check=True)
if len(glob.glob(f"{DATA}/*.npz")) < 567:
    os.makedirs(f"{W}/data", exist_ok=True)
    subprocess.run(["tar", "-xzf", f"{UPLOAD}/msr_npz.tar.gz", "-C", f"{W}/data"], check=True)
n = len(glob.glob(f"{DATA}/*.npz"))
assert n >= 567, f"Only {n} npz files (expected 567)"

PY = sys.executable
ENV = dict(os.environ, PYTHONPATH=f"{CODE}/src", PYTHONDONTWRITEBYTECODE="1", PYTHONUNBUFFERED="1",
           MPLBACKEND="Agg", KCA_EXT=EXT, KCA_PURE_OPS="1")
if os.path.isdir("/usr/local/cuda"):
    ENV.setdefault("CUDA_HOME", "/usr/local/cuda")


def run(cmd, log=None):
    print("$", cmd, flush=True)
    t0, fh = time.time(), (open(f"{LOGS}/{log}", "a") if log else None)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE, env=ENV, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    try:
        for line in p.stdout:
            print(line, end="", flush=True)
            if fh:
                fh.write(line); fh.flush()
        p.wait()
    except KeyboardInterrupt:
        p.kill(); raise
    finally:
        if fh:
            fh.close()
    print(f"[exit {p.returncode} · {(time.time() - t0) / 60:.1f} min]", flush=True)
    if p.returncode:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")


subprocess.run([PY, "-m", "pip", "install", "-q", "einops", "timm", "termcolor", "ninja"], check=True)
print(f"Code: {CODE} | Data: {n} npz files")

Code: /content/kca_seed2/code | Data: 567 npz files


In [4]:
# CUDA fast paths (P4Transformer only uses FPS; without mamba_ssm, selective scan shows off, which does not affect this experiment)
r = subprocess.run([PY, "-c", "from kca.backbones import cpu_ops; cpu_ops.install(); "
                    "from kca.backbones import cuda_fast; import json; print(json.dumps(cuda_fast.STATUS))"],
                   cwd=CODE, env=ENV, capture_output=True, text=True)
print(r.stderr[-2000:])
if r.returncode:
    raise RuntimeError("The fast-path check failed; see the output above")
STATUS = json.loads(r.stdout.strip().splitlines()[-1])
print("FPS kernel:", STATUS["fps"], "| selective scan kernel:", STATUS["scan"])
if STATUS["fps"] != "on":
    print("⚠️ FPS falls back to the step-by-step loop: same results, only slower.")

[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: off (ModuleNotFoundError: No module named 'selective_scan_cuda')

FPS kernel: on | selective scan kernel: off (ModuleNotFoundError: No module named 'selective_scan_cuda')


In [5]:
# Crossover: P4Transformer, seed 2 (same recipe as the main experiment: --scale official --points 2048)
OUT = f"{OUT_ROOT}/crossover_seed{SEED}"
J = f"{OUT}/crossover.json"
if os.path.exists(J) and all("kca" in json.load(open(J)).get(t, {}) for t in ("motion", "geometry")):
    print("Already done, skipping")
else:
    run(f"{PY} -m kca.run.msr_crossover --backbone p4transformer --seed {SEED} --data {DATA} --out {OUT} "
        f"--scale official --points {POINTS} --device cuda --explain-device cuda --reuse",
        log=f"crossover_seed{SEED}.log")

$ /usr/bin/python3 -m kca.run.msr_crossover --backbone p4transformer --seed 2 --data /content/kca_seed2/data/msr --out /content/drive/MyDrive/kca_full/seed2/crossover_seed2 --scale official --points 2048 --device cuda --explain-device cuda --reuse
arm-moving clips: train 163 test 200
[kca.cuda_fast] fps: on (picks equal to the loop's: 100.00%) | scan: off (ModuleNotFoundError: No module named 'selective_scan_cuda')
[motion] trained: test acc 0.677 (104s)
[motion] acc 0.677 | KCA arm motion share 0.728 (ctx 0.744) | motion-prior EBPG(arm) 0.28
[geometry] trained: test acc 0.887 (97s)
[geometry] acc 0.887 | KCA arm motion share 0.574 (ctx 0.570) | motion-prior EBPG(arm) 0.28
[exit 0 · 8.2 min]


In [6]:
# Result summary (copy these lines for the analysis)
c = json.load(open(J))
ok = True
for task, name in (("motion", "Motion task"), ("geometry", "Shape task")):
    acc = 100 * c[task]["test_acc"]
    print(f"{name}: test accuracy {acc:.1f}% | arm motion share {c[task]['kca']['arm_motion_share']:.2f} "
          f"(context {c[task]['kca_ctx']['arm_motion_share']:.2f})")
    ok &= acc >= 60
gap = c.get("ci_kca", {}).get("gap")
if gap:
    print(f"Gap 95% CI: [{gap[0]:.2f}, {gap[1]:.2f}]")
print("✓ Both task models learned their task (≥ 60%)" if ok else "⚠️ A task model is stuck near chance (< 60%); this run cannot be used as is, flag it for review")
print(f"\nOutput directory: {OUT} (the .pth files are here too)")

Motion task: test accuracy 67.8% | arm motion share 0.73 (context 0.74)
Shape task: test accuracy 88.8% | arm motion share 0.57 (context 0.57)
Gap 95% CI: [0.06, 0.26]
✓ Both task models learned their task (≥ 60%)

Output directory: /content/drive/MyDrive/kca_full/seed2/crossover_seed2 (the .pth files are here too)
